<a href="https://colab.research.google.com/github/wasjediknight/papers/blob/main/pncp_contratos_software_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Contratos de desenvolvimento e sustentação de software nos órgãos federais (PNCP)

Este notebook consulta a API pública do PNCP, baixa os contratos mês a mês, mantém só os de **órgãos federais** cujo objeto trate de fábrica de software, desenvolvimento, manutenção ou sustentação de sistemas, pontos de função, UST, perfis profissionais (Portaria 750), métodos ágeis ou Codificagov, e gera um CSV com o link de cada contrato.

**Como usar:** ajuste o período na célula 2 e rode as células em ordem (Ambiente de execução > Executar tudo).

**Observações**
- A API não tem busca por palavra-chave, então todos os contratos do país são percorridos e filtrados aqui. Vários anos podem levar horas.
- Se você salvar no Google Drive (recomendado), a coleta pode ser retomada de onde parou caso o Colab desconecte: basta rodar tudo de novo.
- Contratos antigos da Lei 8.666 geralmente **não** estão no PNCP.

In [8]:
# 1. Parâmetros
INICIO = "2023-01"          # AAAA-MM
FIM    = "2026-09"          # AAAA-MM
SALVAR_NO_DRIVE = True      # recomendado: permite retomar se o Colab cair
TAMANHO_PAGINA = 500        # entre 10 e 500

In [9]:
# 2. Onde salvar
import os
if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/pncp_software'
else:
    PASTA = '/content/pncp_software'
os.makedirs(PASTA, exist_ok=True)
SAIDA = os.path.join(PASTA, 'contratos_software_federal.csv')
PROGRESSO = SAIDA + '.progresso'
print('Arquivos em:', PASTA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arquivos em: /content/drive/MyDrive/pncp_software


In [10]:
# 3. Funções (não precisa editar)
import csv, re, sys, time, unicodedata
from datetime import date, timedelta
import requests

BASE = "https://pncp.gov.br/api/consulta/v1/contratos"
PORTAL = "https://pncp.gov.br/app/contratos/{cnpj}/{ano}/{seq}"

# Termos procurados no objeto do contrato (sem acento, minúsculo)
TERMOS = [
    r"fabrica de software", r"desenvolvimento.{0,40}(software|sistema)",
    r"(manutencao|sustentacao).{0,40}(software|sistema)",
    r"sustentacao de (software|sistema|aplica)", r"ciclo de vida de software",
    r"pontos? de funcao", r"\bust\b", r"unidade de servico tecnico",
    r"perfil profissional de ti", r"portaria sgd/mgi.{0,10}750",
    r"metodos? ageis|praticas ageis|desenvolvimento agil", r"codificagov",
    r"teste.{0,30}qualidade de software",
]
REGEX = re.compile("|".join(TERMOS))

# Nomes para agrupar por ministério (casamento na razão social do órgão)
MINISTERIOS = {
    "Agricultura": "agricultura", "Cidades": "cidades", "MCTI": "ciencia, tecnologia",
    "Comunicações": "comunicacoes", "Cultura": "cultura", "Defesa": "defesa",
    "MDA": "desenvolvimento agrario", "MDS": "desenvolvimento e assistencia social",
    "MDIC": "desenvolvimento, industria", "Direitos Humanos": "direitos humanos",
    "Educação": "educacao", "Empreendedorismo": "empreendedorismo", "Esporte": "esporte",
    "Fazenda": "fazenda", "MGI": "gestao e da inovacao", "Igualdade Racial": "igualdade racial",
    "Integração (MIDR)": "integracao e do desenvolvimento regional", "Justiça": "justica",
    "Meio Ambiente": "meio ambiente", "Minas e Energia": "minas e energia",
    "Mulheres": "mulheres", "Pesca": "pesca", "Planejamento": "planejamento e orcamento",
    "Portos e Aeroportos": "portos e aeroportos", "Povos Indígenas": "povos indigenas",
    "Previdência": "previdencia social", "Relações Exteriores": "relacoes exteriores",
    "Saúde": "saude", "Trabalho": "trabalho e emprego", "Transportes": "transportes",
    "Turismo": "turismo", "AGU": "advocacia-geral", "CGU": "controladoria-geral",
    "Presidência (Casa Civil, SG, SRI, Secom, GSI)": "presidencia da republica",
}

COLUNAS = ["ministerio_provavel", "orgao", "cnpj_orgao", "unidade", "numero_contrato",
           "processo", "fornecedor", "cnpj_fornecedor", "objeto", "valor_global",
           "data_assinatura", "vigencia_inicio", "vigencia_fim", "link_pncp"]


def norm(txt):
    txt = unicodedata.normalize("NFKD", txt or "").encode("ascii", "ignore").decode()
    return txt.lower()


def meses(inicio, fim):
    a, m = map(int, inicio.split("-"))
    fa, fm = map(int, fim.split("-"))
    while (a, m) <= (fa, fm):
        ini = date(a, m, 1)
        prox = date(a + (m == 12), m % 12 + 1, 1)
        yield ini, prox - timedelta(days=1)
        a, m = prox.year, prox.month


def get(params, tentativas=6):
    for t in range(tentativas):
        try:
            r = requests.get(BASE, params=params, timeout=60,
                             headers={"accept": "application/json"})
            # A API sinaliza coleção vazia com 204 ou 404
            if r.status_code in (204, 404):
                return {"data": [], "totalPaginas": 0}
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(5 * (t + 1)); continue
            print(f"  HTTP {r.status_code}: {r.text[:200]}", file=sys.stderr)
            return None
        except requests.RequestException as e:
            print(f"  erro de rede ({e}); tentando de novo", file=sys.stderr)
            time.sleep(5 * (t + 1))
    return None


def eh_federal(c):
    org = c.get("orgaoEntidade") or {}
    if org.get("esferaId"):
        return org["esferaId"] == "F"
    nome = norm(org.get("razaoSocial"))
    return "ministerio" in nome or "presidencia da republica" in nome


def ministerio(nome):
    n = norm(nome)
    for rotulo, chave in MINISTERIOS.items():
        if chave in n:
            return rotulo
    return "Outro órgão federal (verificar vínculo)"


def linha(c):
    org = c.get("orgaoEntidade") or {}
    uni = c.get("unidadeOrgao") or {}
    cnpj = org.get("cnpj", "")
    return {
        "ministerio_provavel": ministerio(org.get("razaoSocial")),
        "orgao": org.get("razaoSocial", ""), "cnpj_orgao": cnpj,
        "unidade": f"{uni.get('codigoUnidade', '')} {uni.get('nomeUnidade', '')}".strip(),
        "numero_contrato": c.get("numeroContratoEmpenho", ""),
        "processo": c.get("processo", ""),
        "fornecedor": c.get("nomeRazaoSocialFornecedor", ""),
        "cnpj_fornecedor": c.get("niFornecedor", ""),
        "objeto": " ".join((c.get("objetoContrato") or "").split()),
        "valor_global": c.get("valorGlobal", ""),
        "data_assinatura": c.get("dataAssinatura", ""),
        "vigencia_inicio": c.get("dataVigenciaInicio", ""),
        "vigencia_fim": c.get("dataVigenciaFim", ""),
        "link_pncp": PORTAL.format(cnpj=cnpj, ano=c.get("anoContrato", ""),
                                   seq=c.get("sequencialContrato", "")),
    }


In [11]:
# 4. Coleta (pode demorar; rode de novo para retomar)
feitos = set(open(PROGRESSO).read().split()) if os.path.exists(PROGRESSO) else set()
novo = not os.path.exists(SAIDA)
vistos = set()
if not novo:
    with open(SAIDA, encoding='utf-8-sig') as f:
        vistos = {r['link_pncp'] for r in csv.DictReader(f, delimiter=';')}

tamanho = TAMANHO_PAGINA
with open(SAIDA, 'a', newline='', encoding='utf-8-sig') as f:
    w = csv.DictWriter(f, fieldnames=COLUNAS, delimiter=';')
    if novo:
        w.writeheader()
    for ini, fim in meses(INICIO, FIM):
        chave = ini.strftime('%Y-%m')
        if chave in feitos:
            print(f'{chave}: já coletado, pulando'); continue
        pagina, total, achados = 1, None, 0
        while total is None or pagina <= total:
            p = {'dataInicial': ini.strftime('%Y%m%d'), 'dataFinal': fim.strftime('%Y%m%d'),
                 'pagina': pagina, 'tamanhoPagina': tamanho}
            res = get(p)
            if res is None and tamanho > 50:
                tamanho = 50; print('  reduzindo tamanho da página para 50'); continue
            if res is None:
                print(f'  falha em {chave} pág. {pagina}; pulando página'); pagina += 1; continue
            total = res.get('totalPaginas') or 0
            for c in res.get('data') or []:
                if eh_federal(c) and REGEX.search(norm(c.get('objetoContrato'))):
                    l = linha(c)
                    if l['link_pncp'] not in vistos:
                        vistos.add(l['link_pncp']); w.writerow(l); achados += 1
            print(f'\r{chave}: página {pagina}/{total} | encontrados no mês: {achados}   ', end='')
            pagina += 1
            time.sleep(0.3)
        f.flush()
        with open(PROGRESSO, 'a') as pf:
            pf.write(chave + '\n')
        print(f'\r{chave}: concluído, {achados} contratos encontrados' + ' ' * 20)
print('Pronto:', SAIDA)

2023-01: já coletado, pulando
2023-02: já coletado, pulando
2023-03: já coletado, pulando
2023-04: já coletado, pulando
2023-05: já coletado, pulando
2023-06: já coletado, pulando
2023-07: já coletado, pulando
2023-08: já coletado, pulando
2023-09: já coletado, pulando
2023-10: já coletado, pulando
2023-11: já coletado, pulando
2023-12: já coletado, pulando
2024-01: já coletado, pulando
2024-02: já coletado, pulando
2024-03: já coletado, pulando
2024-04: já coletado, pulando
2024-05: já coletado, pulando
2024-06: já coletado, pulando
2024-07: já coletado, pulando
2024-08: já coletado, pulando
2024-09: já coletado, pulando
2024-10: já coletado, pulando
2024-11: já coletado, pulando
2024-12: já coletado, pulando
2025-01: já coletado, pulando
2025-02: já coletado, pulando
2025-03: já coletado, pulando
2025-04: já coletado, pulando
2025-05: já coletado, pulando
2025-06: já coletado, pulando
2025-07: já coletado, pulando
2025-08: já coletado, pulando
2025-09: já coletado, pulando
2025-10: j

In [12]:
# 5. Resumo por ministério
import pandas as pd
df = pd.read_csv(SAIDA, sep=';', encoding='utf-8-sig')
df['valor_global'] = pd.to_numeric(df['valor_global'], errors='coerce')
resumo = (df.groupby('ministerio_provavel')
            .agg(contratos=('link_pncp', 'count'), valor_total=('valor_global', 'sum'),
                 fornecedores=('fornecedor', lambda s: ', '.join(sorted(set(s.dropna()))[:5])))
            .sort_values('contratos', ascending=False))
pd.set_option('display.max_colwidth', 120)
print(f'{len(df)} contratos encontrados')
resumo

2405 contratos encontrados


,contratos,valor_total,fornecedores
ministerio_provavel,,,
Outro órgão federal (verificar vínculo),1618,6.793775e+09,"12.921.767 JOAO LUIZ BENTO E SILVA, 21.595.867 IRANALDO SANTOS ARAUJO, 23.021.050 THAIS MORET MARACCINI, 23.727.248 ..."
Educação,229,3.080495e+07,"11.883.438 KAYTO WEREZ SECCO, 17.171.634 DANIEL FRANCISCO KLEIN, 27.999.909 ANTONIO JOSE SARAIVA DA SILVA, 29.584.75..."
Justiça,160,5.299811e+08,"49.009.169 LUIZ HENRIQUE DE BARROS SANTOS, 49.432.007 EDUARDO DA SILVA DOS SANTOS, 51.609.577 MAURO EDER DO NASCIMEN..."
Saúde,131,6.854349e+07,"A A COSTA CONSTRUCOES LTDA, A C P DA SILVA QUINOY COMERCIO E SERVICOS, A S GODINHO TINTAS, A. L. M.T. - ADMINISTRAC..."
Fazenda,67,1.329310e+08,"A. DA SILVA CORREA, AGIL LTDA, ARCA EMPREENDIMENTOS COMERCIO E SERVICOS LTDA, ARP MANUTENCAO DE EQUIPAMENTOS ELETRIC..."
MCTI,40,2.725760e+06,"50.407.086 ERICO FELIPE SOUZA RIBEIRO, 60.988.534 EVERTON CANDIDO LOPES, A C P DA SILVA QUINOY COMERCIO E SERVICOS,..."
Agricultura,30,4.475345e+07,"54.132.859 THIAGO ACCIOLY MARINHO, AZZELAB COMERCIO DE EQUIPAMENTOS LTDA, FERRI COMERCIO E SERVICOS LTDA, GRAULAB BR..."
Transportes,16,1.918460e+08,"AUDAZ SISTEMAS ELETRONICOS DE SEGURANCA LTDA, BLUE - SOLUCOES INTELIGENTES LTDA, DELTALAB CONSULTORIA E TREINAMENTOS..."
Meio Ambiente,16,5.377918e+07,"BASIS TECNOLOGIA DA INFORMACAO S.A., EMPRESA DE TECNOLOGIA E INFORMACOES DA PREVIDENCIA - DATAPREV S.A., FIRST POINT..."


In [13]:
# 6. Ver os contratos (filtre pelo ministério se quiser)
MINISTERIO = ''   # ex.: 'Educação'; vazio mostra todos
v = df if not MINISTERIO else df[df['ministerio_provavel'] == MINISTERIO]
v[['ministerio_provavel', 'orgao', 'fornecedor', 'objeto', 'valor_global', 'vigencia_fim', 'link_pncp']]

,ministerio_provavel,orgao,fornecedor,objeto,valor_global,vigencia_fim,link_pncp
0,Educação,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNOLOGIA DE MINAS GERAIS",INFOCENTER LTDA,"CONTRATAÇÃO DE SERVIÇO MANUTENÇÃO DE SOFTWARE E SUPORTE TÉCNICO MENSAL PARA MANUTENÇÃO DOS SISTEMAS DE CONTROLE, SR-...",7999.92,2023-12-31,https://pncp.gov.br/app/contratos/10626896000172/2022/837
1,Trabalho,MINISTERIO DO TRABALHO E EMPREGO - MTE,IMPLANTA INFORMATICA LTDA,O presente Contrato tem por objeto a aquisição e prestação de serviços contínuos de suporte e manutenção de sistema ...,884574.00,2027-12-29,https://pncp.gov.br/app/contratos/37115367000160/2022/2
2,MCTI,"MINISTERIO DA CIENCIA, TECNOLOGIA E INOVACOES",ASSOCIACAO PARANAENSE DE CULTURA - APC,PRESTAÇÃO DE SERVIÇO DE MANUTENÇÃO DO SISTEMA PERGAMUM.,10976.64,2023-10-30,https://pncp.gov.br/app/contratos/01263896000164/2022/1025
3,Justiça,MINISTERIO DA JUSTICA E SEGURANCA PUBLICA,BRASIL FIRE EQUIPAMENTOS DE COMBATE A INCENDIO LTDA,O OBJETO DO PRESENTE INSTRUMENTO É A CONTRATAÇÃO DE EMPRESA ESPECIALIZADA NA MANUTENÇÃO DE SISTEMA DE COMBATE A INCÊ...,8000.00,2025-07-06,https://pncp.gov.br/app/contratos/00394494010441/2022/268
4,Educação,INSTITUTO FEDERAL DE EDUCACAO CIENCIA E TECNOLOGIA DO ESPIRITO SANTO,ELETROLINKI EXECUCOES E MANUTENCOES LTDA,"DISPENSA DE LICITAÇÃO 41/2022. PROC. 23185.002043/2022-28. CONTRATAÇÃO DE SERVIÇOS TÉCNICOS PROFISSIONAIS, SERVIÇO N...",4850.00,2023-12-31,https://pncp.gov.br/app/contratos/10838653000106/2023/1
...,...,...,...,...,...,...,...
2400,Outro órgão federal (verificar vínculo),UNIVERSIDADE FEDERAL DE CATALAO,INSTASOLUTIONS PRODUTOS E GESTAO EMPRESARIAL LTDA,"CONTRATAÇÃO DE SERVIÇOS COMUNS DE SOLUÇÃO INTEGRADA PARA ABASTECIMENTO E MANUTENÇÃO DE VEÍCULOS, POR MEIO DE SISTEMA...",489004.26,2027-10-01,https://pncp.gov.br/app/contratos/35834377000120/2026/92
2401,Outro órgão federal (verificar vínculo),MINISTERIO PUBLICO DA UNIAO,GALVAO SERVICOS DE CLIMATIZACAO LTDA,Contratação de empresa especializada para a prestação de serviços continuados de manutenção preventiva e corretiva d...,8400.00,2027-09-23,https://pncp.gov.br/app/contratos/26989715000102/2026/2003
2402,Outro órgão federal (verificar vínculo),MINISTERIO PUBLICO DA UNIAO,J P BARBOSA E SILVA LTDA,Prestação de serviços continuados de manutenção dos equipamentos do sistema de combate a incêndio na sede da Procura...,1192.00,2026-09-30,https://pncp.gov.br/app/contratos/26989715000102/2025/3823
2403,Educação,INSTITUTO FEDERAL DE EDUCACAO CIENCIA E TECNOLOGIA DO ESPIRITO SANTO,ELIAH PROJETOS E SERVICOS ELETRICOS LTDA,CONTRATAÇÃO DE SERVIÇOS DE MANUTENÇÃO PREVENTIVA DOS SISTEMAS DE GERAÇÃO FOTOVOLTAICA DO IFES CAMPUS VENDA NOVA DO I...,5980.00,2026-12-31,https://pncp.gov.br/app/contratos/10838653000106/2026/649


In [14]:
# 7. Baixar o CSV e o resumo
resumo.to_csv(os.path.join(PASTA, 'resumo_por_ministerio.csv'), sep=';', encoding='utf-8-sig')
from google.colab import files
files.download(SAIDA)
files.download(os.path.join(PASTA, 'resumo_por_ministerio.csv'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>